# Task 3: Supervised fine-tuning

**Settings:** Accelerator **GPU T4 x2**, Internet **on**. Run with **Save Version > Save & Run All (Commit)**.

1. Prepare 30,000 training and 1,000 validation pairs from UltraChat 200k (first user turn and first
   reply, at most 512 tokens with EOS, evaluation prompts removed).
2. A 20-step timing run checks memory and projects the full run time; the notebook stops if either is out of bounds.
3. Full SFT of `Qwen/Qwen2.5-0.5B` on GPU 0 with the hand-written loop (`src/pa2/train_sft.py`, `configs/sft.yaml`).
4. Check that validation loss improved, plot the curves.
5. Decoding check: the best SFT checkpoint answers 50 held-out validation prompts with plain greedy (GPU 0) and
   greedy with repetition penalty 1.1 (GPU 1). This picks the decoding setting for Task 6.

Checkpoints stay in this notebook's output (`outputs/sft/run/best`, `outputs/sft/run/final`) for Task 4.
The small files (metrics, logs, plot, statistics, decoding check) are archived as `task3_sft_outputs.zip`.

In [ ]:
REPO_URL = "https://github.com/mabdullahi7780/small-model-alignment-starter.git"
BRANCH = "main"
WORK_DIR = "/kaggle/working"
SCRATCH_DIR = "/tmp"
REPO_DIR = f"{SCRATCH_DIR}/small-model-alignment-starter"
OUT_DIR = f"{WORK_DIR}/outputs/sft"
CONFIG_FILE = "configs/sft.yaml"
DATA_DIR = f"{OUT_DIR}/data"
RUN_DIR = f"{OUT_DIR}/run"
BASE_MODEL = "Qwen/Qwen2.5-0.5B"
SMOKE_STEPS = 20
MAX_PEAK_GIB = 14.0     # a T4 has 14.56 GiB
MAX_HOURS = 4.0
RUN_SETTINGS = {"config": CONFIG_FILE, "smoke_steps": SMOKE_STEPS, "max_peak_gib": MAX_PEAK_GIB, "max_hours": MAX_HOURS}

In [ ]:
import os
import subprocess
import sys


def run(cmd, cwd=None, env=None, check=True):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)


if not os.path.isdir(REPO_DIR):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
run(["git", "log", "-1", "--format=commit %H%nauthor date %ad%nsubject %s"], cwd=REPO_DIR)

In [ ]:
# Kaggle's preinstalled torch is built for its GPU runtime; pin it so pip can never replace it.
TORCH_BEFORE = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__)"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print("Preinstalled torch:", TORCH_BEFORE)

constraint_file = f"{SCRATCH_DIR}/torch-constraint.txt"
with open(constraint_file, "w") as handle:
    handle.write(f"torch=={TORCH_BEFORE}\n")

run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-c", constraint_file], cwd=REPO_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=REPO_DIR)

In [ ]:
import json

COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
os.makedirs(f"{OUT_DIR}/logs", exist_ok=True)
with open(f"{OUT_DIR}/run_info.json", "w") as handle:
    json.dump({"commit": COMMIT, "torch": TORCH_BEFORE, **RUN_SETTINGS}, handle, indent=2)
print("commit", COMMIT)

In [ ]:
def run_on_gpus(jobs, tag):
    """Start one subprocess per job, each pinned to its own GPU, and wait for all of them.

    jobs: list of (name, gpu_index, argv). Each job's output goes to logs/<tag>_<name>.log.
    """
    env_base = {**os.environ, "PYTHONPATH": f"{REPO_DIR}/src:{REPO_DIR}"}
    running = []
    for name, gpu, argv in jobs:
        log_path = f"{OUT_DIR}/logs/{tag}_{name}.log"
        handle = open(log_path, "w")
        print(f"start {tag}/{name} on GPU {gpu}", flush=True)
        process = subprocess.Popen(
            [sys.executable, *argv], cwd=REPO_DIR, stdout=handle, stderr=subprocess.STDOUT,
            env={**env_base, "CUDA_VISIBLE_DEVICES": str(gpu)},
        )
        running.append((name, process, handle, log_path))
    failed = []
    for name, process, handle, log_path in running:
        code = process.wait()
        handle.close()
        print(f"\n===== {tag}/{name}: exit code {code} =====")
        print(open(log_path).read()[-8000:])
        if code:
            failed.append(name)
    if failed:
        raise RuntimeError(f"{tag}: failed jobs {failed}; see {OUT_DIR}/logs")

## 1. Data: UltraChat 200k to single-turn pairs

In [ ]:
run_on_gpus([("prep", "", ["-m", "pa2.sft_data", "--config", CONFIG_FILE, "--out-dir", DATA_DIR])], "data")

## 2. Timing run: 20 optimizer steps, no checkpoints

If it runs out of memory or exceeds the limit, it is retried once with gradient checkpointing, and the full run then uses that setting.

In [ ]:
def timing_run(extra, tag):
    try:
        run_on_gpus([(tag, 0, ["-m", "pa2.train_sft", "--config", CONFIG_FILE, "--data-dir", DATA_DIR,
                               "--out-dir", f"{OUT_DIR}/{tag}", "--max-steps", str(SMOKE_STEPS), "--no-save", *extra])], tag)
    except RuntimeError as error:
        print("timing run failed:", error)
        return None
    return json.load(open(f"{OUT_DIR}/{tag}/summary.json"))


EXTRA = []
smoke = timing_run(EXTRA, "smoke")
if smoke is None or smoke.get("peak_reserved_gib", 0) > MAX_PEAK_GIB:
    EXTRA = ["--set", "gradient_checkpointing=true"]
    print("Retrying with gradient checkpointing")
    smoke = timing_run(EXTRA, "smoke_checkpointing")
    if smoke is None:
        raise RuntimeError("timing run failed even with gradient checkpointing")
print(json.dumps(smoke, indent=2))
if smoke.get("peak_reserved_gib", 0) > MAX_PEAK_GIB:
    raise RuntimeError(f"peak memory {smoke['peak_reserved_gib']} GiB is above {MAX_PEAK_GIB}")
if smoke["projected_full_run_hours"] > MAX_HOURS:
    raise RuntimeError(f"projected {smoke['projected_full_run_hours']} h is above {MAX_HOURS} h")
if smoke["skipped_steps"] > SMOKE_STEPS // 2:
    raise RuntimeError("most steps were skipped by the loss scaler: fp16 overflow")
print("Timing run within bounds, starting the full run.")

## 3. Full SFT run

In [ ]:
print("extra settings:", EXTRA or "none")
run_on_gpus([("train", 0, ["-m", "pa2.train_sft", "--config", CONFIG_FILE, "--data-dir", DATA_DIR,
                           "--out-dir", RUN_DIR, *EXTRA])], "train")

## 4. Did it learn? Curves

In [ ]:
summary = json.load(open(f"{RUN_DIR}/summary.json"))
print(json.dumps(summary, indent=2))
assert os.path.exists(f"{RUN_DIR}/best/model.safetensors") and os.path.exists(f"{RUN_DIR}/final/model.safetensors")
if not summary["best_val_loss"] < summary["initial_val_loss"]:
    raise RuntimeError("validation loss never improved on the base model")

run([sys.executable, "-m", "pa2.plots", "sft", "--metrics", f"{RUN_DIR}/metrics.jsonl", "--out", f"{OUT_DIR}/sft_curves.png"],
    cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": f"{REPO_DIR}/src:{REPO_DIR}"})
from IPython.display import Image, display
display(Image(f"{OUT_DIR}/sft_curves.png"))

## 5. Decoding check on 50 held-out validation prompts

In [ ]:
CHECK_PROMPTS = f"{DATA_DIR}/decoding_check_prompts.json"
VARIANTS = {"greedy": (0, 1.0), "greedy_rp1.1": (1, 1.1)}
run_on_gpus([
    (name, gpu, ["-m", "pa2.generate", "--model", f"{RUN_DIR}/best", "--tokenizer", BASE_MODEL,
                 "--generator", "qwen2.5-0.5b-sft", "--prompts", CHECK_PROMPTS,
                 "--out", f"{OUT_DIR}/decoding_check/sft_{name}.json", "--batch-size", "25",
                 "--max-new-tokens", "512", "--repetition-penalty", str(penalty), "--show", "2"])
    for name, (gpu, penalty) in VARIANTS.items()
], "decoding_check")

table = {name: json.load(open(f"{OUT_DIR}/decoding_check/sft_{name}.meta.json"))["stats"] for name in VARIANTS}
json.dump(table, open(f"{OUT_DIR}/decoding_check/comparison.json", "w"), indent=2)
print(f"{'metric':<22}" + "".join(f"{name:>16}" for name in table))
for key in next(iter(table.values())):
    print(f"{key:<22}" + "".join(f"{table[name][key]:>16.3f}" for name in table))

outputs = {name: json.load(open(f"{OUT_DIR}/decoding_check/sft_{name}.json")) for name in VARIANTS}
for index in (0, 1, 2):
    print("\n" + "=" * 100 + "\nPROMPT:", outputs["greedy"][index]["instruction"][:300])
    for name in VARIANTS:
        record = outputs[name][index]
        print(f"\n--- {name} [{record['stop_reason']}, {record['new_tokens']} tokens]\n{record['output'][:700]}")

## Archive the small outputs

In [ ]:
import zipfile


def zip_small_outputs(archive_path, root, skip_dirs=(), skip_suffixes=(".safetensors", ".jsonl.gz")):
    """Zip everything under root except checkpoints and bulky data, so the archive stays small."""
    count = 0
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for folder, dirs, files in os.walk(root):
            rel_folder = os.path.relpath(folder, root)
            dirs[:] = [d for d in dirs if os.path.normpath(os.path.join(rel_folder, d)) not in skip_dirs]
            for name in sorted(files):
                if name.endswith(skip_suffixes):
                    continue
                path = os.path.join(folder, name)
                archive.write(path, os.path.relpath(path, root))
                count += 1
    print(f"Saved {archive_path} ({count} files, {os.path.getsize(archive_path) / 1e6:.1f} MB)")


zip_small_outputs(f"{WORK_DIR}/task3_sft_outputs.zip", OUT_DIR,
                  skip_dirs=("run/best", "run/final"), skip_suffixes=(".safetensors", "train.jsonl"))